# 03 — Benchmark models

Five benchmarks under the rolling-origin protocol.

**Report sections fed:** 5 (Benchmark models).


In [ ]:
import sys
sys.path.insert(0, "../src")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from appliance_energy import config, data, evaluation, features, plotting, stationarity
from appliance_energy.models import benchmarks, feature_models, foundation, sarimax

pd.set_option("display.width", 140)


In [ ]:
frame = data.load_hourly()
y = frame[config.TARGET]

y_train, y_test = data.train_test_split(y)
test_index = y_test.index

print(f"train {y_train.index.min()} -> {y_train.index.max()}  ({len(y_train)})")
print(f"test  {test_index.min()} -> {test_index.max()}  ({len(y_test)})")


## Rolling-origin protocol

Fourteen origins spaced 24 hours apart. At each origin a model sees everything
strictly before the origin — including test observations released by earlier
blocks — and forecasts 24 steps.


In [ ]:
suite = benchmarks.benchmark_suite(config.DAILY_PERIOD, config.WEEKLY_PERIOD)

forecasts = {
    name: benchmarks.rolling_origin_forecast(y, test_index, config.HORIZON, fn)
    for name, fn in suite.items()
}

evaluation.evaluate_all(forecasts, y_test, y_train).round(3)


### Error by lead time\n\nWhich benchmarks decay away from the origin, and which are flat?

In [ ]:
lead = evaluation.errors_by_horizon(forecasts, y_test, config.HORIZON)
fig = plotting.plot_error_by_lead_time(lead)


### Sanity check on the protocol

Perturbing the first block's actuals must not change the first block's forecast,
but must change later blocks. This mirrors `tests/test_benchmarks.py`.


In [ ]:
tampered = y.copy()
tampered.loc[test_index[:24]] += 5000

base = benchmarks.rolling_origin_forecast(y, test_index, 24, benchmarks.naive_forecast)
alt = benchmarks.rolling_origin_forecast(tampered, test_index, 24, benchmarks.naive_forecast)

print("block 1 unchanged:", np.allclose(base.iloc[:24], alt.iloc[:24]))
print("block 2 changed:  ", not np.allclose(base.iloc[24:48], alt.iloc[24:48]))
